In [2]:
%load_ext autoreload
%autoreload 2

import ezy_seq as ezy
#import ezy_seq.load 
import scanpy as sc
import pandas as pd
import os
from pathlib import Path  # For file path operations
import matplotlib.pyplot as plt
import napari
import glob
import numpy as np
import pathlib
from matplotlib.path import Path as MPath  # For polygon operations (matplotlib Path)
import anndata as ad
from scipy import sparse
import random
random.seed(0)


c:\Users\woods\miniconda3\envs\new_env\lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
c:\Users\woods\miniconda3\envs\new_env\lib\site-packages\anndata\utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


Load CosMx Exports directly into AnnData object

In [ ]:
r"""First, Load "polygons.csv.gz" files for cellwise labeling in Napari"""
cell_type_col_name='cell_type_name'
file_path = r"/path/to/sample_info.xlsx"# Csv with samplewise metadata, not requried, just allows more rapid metadata inclusion.
meta_dictionary=ezy.read_dictionary(file_path)
CosMx_Export_Path=r"path/to/CosMx/Export"
adatas=ezy.load.cosmx(CosMx_Export_Path)



################################################################################################################################
def load_polys(fp):
    if not os.path.exists(fp): return []
    df = pd.read_csv(fp)
    axes = ['axis-0','axis-1']
    xcol,ycol = axes[1], axes[0]
    df = df.dropna(subset=[xcol,ycol])
    return [g[[ycol,xcol]].to_numpy() for _,g in df.groupby('index')]

def load_all_polygons(project_folder):
    project_folder_ = pathlib.Path(project_folder)
    polygons = {}
    for fp in project_folder_.rglob("*-polygons.csv.gz"):
        slide_name_ = fp.stem.replace("-polygons", "")
        slide_name = slide_name_.replace(".csv", "")
        polygons[slide_name] = pd.read_csv(fp)
    return polygons

project_folder = CosMx_Export_Path
polygons_dict = load_all_polygons(project_folder)

layer_folders=['120Land323DY', '367Rand215L', '217Rand144_2L', '368R', '118Band1182L']
polygons=[]
for folder in layer_folders:
    polygons.append(polygons_dict[folder])
################################################################################################################################


In [6]:
polygons_dict.keys()

dict_keys(['120Land323DY', '367Rand215L', '217Rand144_2L', '368R', '118Band1182L'])

In [ ]:
r"""Open slides in Napari to generate sample-defining .csvs (only necassary if multiple samples are per slide)."""
import imageio.v2 as imageio
import napari
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Base directory for your project

for slide in adatas:
    # Create subset for the current slide
    s_df = slide

    print(f"Processing slide: {slide.obs['slide_ID'][0]}")
    
    # --- PREPARE COLORS ---
    unique_clusters = s_df.obs['cell_type'].unique()
    cmap = plt.get_cmap('tab20') 
    colors_list = [cmap(i) for i in np.linspace(0, 1, len(unique_clusters))]
    
    cluster_color_dict = dict(zip(unique_clusters, colors_list))
    assigned_colors = s_df.obs['cell_type'].astype(object).map(cluster_color_dict).tolist()

    # --- INITIALIZE VIEWER & ADD POINTS ---
    viewer = napari.Viewer()
    viewer.add_points(
        s_df.obsm['spatial_fov'], 
        size=100,
        border_width=0.0,
        face_color=assigned_colors,
        name="CosMx cells"
    )
napari.run()

C:\Users\woods\AppData\Local\Temp\ipykernel_35492\2383845099.py:16: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"Processing slide: {slide.obs['slide_ID'][0]}")


Processing slide: 120Land323DY


C:\Users\woods\AppData\Local\Temp\ipykernel_35492\2383845099.py:16: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"Processing slide: {slide.obs['slide_ID'][0]}")


Processing slide: 367Rand215L


C:\Users\woods\AppData\Local\Temp\ipykernel_35492\2383845099.py:16: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"Processing slide: {slide.obs['slide_ID'][0]}")


Processing slide: 217Rand144_2L


C:\Users\woods\AppData\Local\Temp\ipykernel_35492\2383845099.py:16: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"Processing slide: {slide.obs['slide_ID'][0]}")


Processing slide: 368R


C:\Users\woods\AppData\Local\Temp\ipykernel_35492\2383845099.py:16: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"Processing slide: {slide.obs['slide_ID'][0]}")


Processing slide: 118Band1182L


In [ ]:
r"""Use Napari-generated .csv files to define samples (only necassary if multiple samples are per slide)."""
processed_adatas = []
base=r"/path/to/data"
# Zip the anndata objects with their corresponding folder
for adata_full in adatas:
    if adata_full.obs['slide_ID'].unique().size>1:
        print('we have multiple slides in this object')
        continue
    slide_folder=adata_full.obs['slide_ID'].unique()[0]
    print(f"Processing slide folder: {slide_folder}")
    s_adata = adata_full.copy()
    s_adata.obs['sample_ID'] = 'Unassigned'
    

    cell_xy = np.column_stack((s_adata.obsm['spatial_fov'][:, 1], s_adata.obsm['spatial_fov'][:, 0]))
    search_path = os.path.join(base, slide_folder, "*.csv")
    print('##############################################################')
    print(search_path)
    print('##############################################################')

    found_csvs = glob.glob(search_path)
    
    if not found_csvs:
        print(f"  No CSVs found in {slide_folder}")
    
    for fp in found_csvs:
        # Extract filename to use as Sample ID
        
        file_name = os.path.basename(fp)
        sample_code = os.path.splitext(file_name)[0]
        if not sample_code[0].isdigit():
            print(f"  -> Skipping {sample_code} ")
            continue
        try:
            df = pd.read_csv(fp)
        except Exception as e:
            print(f"  Error reading {fp}: {e}")
            continue

        # Detect axis columns
        xcol = 'axis-1' if 'axis-1' in df.columns else None
        ycol = 'axis-0' if 'axis-0' in df.columns else None

        if xcol is None or ycol is None:
            print(f"  Bad cols in {fp}; need axis-0/axis-1")
            continue
        hit_mask = np.zeros(len(cell_xy), dtype=bool)
        
        for _, g in df.groupby('index'):
            verts = np.column_stack((g[xcol].to_numpy(dtype=float), g[ycol].to_numpy(dtype=float)))
            if len(verts) < 3: 
                continue
            p = MPath(verts)
            hit_mask |= p.contains_points(cell_xy)

        if hit_mask.any():
            s_adata.obs.loc[hit_mask, 'sample_ID'] = sample_code
            print(f"  Assigned '{sample_code}': {hit_mask.sum()} cells")
    
    print(f"Finished {slide_folder} — Distribution:\n{s_adata.obs['sample_ID'].value_counts()}\n")
    processed_adatas.append(s_adata)

# Merge all slides into one object
adata_full_pre = sc.concat(processed_adatas, joifn='outer', axis=0, merge="first", index_unique='-')



In [ ]:
#combinining

annotate_adatas=ezy.apply_annotation(processed_adatas,meta_dictionary,cell_type_key='RNA_CosMx_Profile_Cell.Typing.InSituType.1_1_clusters')
norm_adatas=ezy.filter_and_normalize(annotate_adatas
    ,min_gene_cnt=15
    ,min_t_cnt=20)
adata_full=sc.concat(norm_adatas, join='outer', axis=0, merge="first", index_unique='-')

adata_full.obs.rename(columns={cell_type_col_name: "cell_type"}, inplace=True)

In [ ]:
r"""Use Napari-generated .csv files to label regions."""

import os
import glob
import pandas as pd
import numpy as np
import scanpy as sc
from matplotlib.path import Path

base=r"/path/to/folder/containing/slide_folders/with/sample_ID/and/region/csvs/defined/in/napari"
dfs = []
# IMPORTANT: Order matters! Regions listed later will OVERRIDE earlier ones.
# Prefrontal is placed AFTER Cortex so it takes priority for overlapping cells.
r'''Below are the regions used in the mansucript, edit as necassary to the name of csvs saved in Napari.'''
categories=['Cerebellum_Cortex.csv', 'Cerebellum_Tract.csv', 'Cortex.csv', 'Hippocampus.csv', 'Olfactory.csv', 'Unassigned.csv',  'Subiculum.csv','Striatum.csv', 'Prefrontal.csv']

# Loop through your sample lists and corresponding folders
for slide_folder in adata_full.obs['slide_ID'].unique():
    
    # Create the subset for the current slide/batch
    s_df = adata_full[adata_full.obs['slide_ID']==slide_folder].copy()
    
    s_df.obs['napari_region'] = 'Unassigned'

    cell_xy = s_df.obsm['spatial_fov'][:, :2] 

    slide_path = os.path.join(base, slide_folder)

    for cat_file in categories:
        fp = os.path.join(slide_path, cat_file)

        # Skip if file doesn't exist for this slide
        if not os.path.exists(fp):
            continue

        region_name = os.path.splitext(cat_file)[0]

        df = pd.read_csv(fp)
        # Detect axis columns: axis-1 is X, axis-0 is Y in Napari
        xcol = 'axis-0' if 'axis-0' in df.columns else None
        ycol = 'axis-1' if 'axis-1' in df.columns else None
        
        if xcol is None or ycol is None:
            print(f"Skipping {fp}: columns need to be axis-0 and axis-1")
            continue

        # Group vertices by polygon index and test points
        hit_mask = np.zeros(len(cell_xy), dtype=bool)
        
        for _, g in df.groupby('index'):
            # Stack as (X, Y) to match cell_xy
            verts = np.column_stack((g[xcol].to_numpy(dtype=float), g[ycol].to_numpy(dtype=float)))
            if len(verts) < 3:  # skip degenerate polygons
                continue
            p = Path(verts)
            # Update mask: true if point is in ANY of the polygons for this region type
            hit_mask |= p.contains_points(cell_xy)

        # Assign the region name ONLY to the 'napari_region' column
        if hit_mask.any():
            s_df.obs.loc[hit_mask, 'napari_region'] = region_name
            print(f"  -> Assigned '{region_name}': {hit_mask.sum()} cells")

    # Append processed subset to list
    print(f"Finished {slide_folder} — Region distribution:\n{s_df.obs['napari_region'].value_counts()}\n")
    dfs.append(s_df)

# Concatenate back together
adata_full = sc.concat(dfs, join='outer', axis=0, merge="first", index_unique='-')
# Save

adata_full.write_h5ad(r"/path/store/adata.h5ad")


Export For QUINT labeling

In [ ]:


def export_anndata_for_seurat(
    adata,
    output_dir,
    *,
    counts_layer_candidates=("counts", "raw"),
    x_filename="normalized_counts.csv",
    counts_filename="raw_counts.csv",
    var_filename="features_counts.csv",
    obs_filename="cell_metadata.csv",
    coords_key="spatial_fov",
    coords_filename="coords_xy.csv",
    pca_key="X_pca",
    pca_filename="pca.csv",
    umap_key="X_umap",
    umap_filename="umap.csv",
    export_other_obsm=True,
    other_obsm_exclude=("spatial_fov", "X_pca", "X_umap"),
    verbose=True,
):
    """
    Export an AnnData object to CSVs for reconstruction in R/Seurat.

    Writes normalized_counts.csv, raw_counts.csv, features_counts.csv,
    cell_metadata.csv, coords_xy.csv, pca.csv, umap.csv, and any other
    obsm matrices as <key>.csv.  Returns a dict of written file paths.
    """
    import os
    import numpy as np
    import pandas as pd
    from scipy import sparse

    os.makedirs(output_dir, exist_ok=True)
    written = {}

    def _dense(mat):
        if sparse.issparse(mat): return mat.toarray()
        if hasattr(mat, "toarray"): return mat.toarray()
        return np.asarray(mat)

    # Raw counts
    layer = next((l for l in counts_layer_candidates if l in getattr(adata, "layers", {})), None)
    counts = adata.layers[layer] if layer else adata.X
    if layer is None and verbose:
        print("Warning: no counts layer found; using X for raw counts")
    fp = os.path.join(output_dir, counts_filename)
    pd.DataFrame(_dense(counts), index=adata.obs_names, columns=adata.var_names).to_csv(fp)
    written["counts"] = fp

    # Normalized (X)
    fp = os.path.join(output_dir, x_filename)
    pd.DataFrame(_dense(adata.X), index=adata.obs_names, columns=adata.var_names).to_csv(fp)
    written["X"] = fp

    # Metadata
    for name, fname in [(adata.var, var_filename), (adata.obs, obs_filename)]:
        fp = os.path.join(output_dir, fname)
        name.to_csv(fp)
        written[fname] = fp

    # Spatial coords
    if coords_key in getattr(adata, "obsm", {}):
        fp = os.path.join(output_dir, coords_filename)
        coords = np.asarray(adata.obsm[coords_key])[:, :2]
        pd.DataFrame(coords, index=adata.obs_names, columns=["x", "y"]).to_csv(fp)
        written["coords"] = fp

    # PCA / UMAP
    for key, fname, col_fmt in [(pca_key, pca_filename, "PC{}"), (umap_key, umap_filename, "UMAP_{}")]:
        if key in getattr(adata, "obsm", {}):
            arr = np.asarray(adata.obsm[key])
            cols = [col_fmt.format(i + 1) for i in range(arr.shape[1])]
            fp = os.path.join(output_dir, fname)
            pd.DataFrame(arr, index=adata.obs_names, columns=cols).to_csv(fp)
            written[key] = fp

    # Any other obsm
    if export_other_obsm:
        for key in adata.obsm_keys():
            if key in other_obsm_exclude: continue
            arr = np.asarray(adata.obsm[key])
            if arr.ndim != 2 or arr.shape[0] != adata.n_obs: continue
            fp = os.path.join(output_dir, f"{key}.csv")
            pd.DataFrame(arr, index=adata.obs_names,
                         columns=[f"{key}_{i+1}" for i in range(arr.shape[1])]).to_csv(fp)
            written[f"obsm:{key}"] = fp

    if verbose:
        print(f"Export complete: {len(written)} files written to {output_dir}")
    return written


In [ ]:
output_dir = "./seurat_export"  # Change to your desired output directory
written_files = export_anndata_for_seurat(adata_full, output_dir)


In [ ]:
#retreiving quint labels from R
quint_df=pd.read_csv(r"path/to/QUINT_csv",index_col=0)
adata_full.obs['quint_region']=quint_df['quint_region']


In [ ]:
r'''Verify Validity of QUINT labels'''

base=r'/path/to/data'
for slide in adata_full.obs['slide_ID'].unique():
    # Create subset for the current slide
    s_df = adata_full[adata_full.obs['slide_ID'] == slide].copy()

    print(f"Processing slide: {slide}")
    
    # --- PREPARE COLORS  ---
    color_by='cell_type' #This can be changed to any cellwise-data useful for visualization.
    unique_clusters = s_df.obs[color_by].unique()
    cmap = plt.get_cmap('tab20') 
    colors_list = [cmap(i) for i in np.linspace(0, 1, len(unique_clusters))]
    
    cluster_color_dict = dict(zip(unique_clusters, colors_list))
    assigned_colors = s_df.obs[color_by].astype(object).map(cluster_color_dict).tolist()

    # --- INITIALIZE VIEWER & ADD POINTS ---
    viewer = napari.Viewer()

    # Base cells
    viewer.add_points(
        s_df.obsm['spatial_fov'], 
        size=100,
        border_width=0.0,
        face_color=assigned_colors,
        name=slide
    )
    
    # --- Add Napari Polygons if Applicable ---
    slide_folder_path = os.path.join(base, slide)
    search_path = os.path.join(slide_folder_path, "*.csv")
    found_csvs = glob.glob(search_path)
    
    if not found_csvs:
        print(f"  -> No polygon CSVs found in {slide_folder_path}")
    
    for fp in found_csvs:
        region_name = os.path.splitext(os.path.basename(fp))[0]
        df = pd.read_csv(fp)
        xcol = 'axis-0' if 'axis-0' in df.columns else None
        ycol = 'axis-1' if 'axis-1' in df.columns else None
        if xcol and ycol:
            polygons_list = []
            for _, g in df.groupby('index'):
                verts = np.column_stack((g[xcol].to_numpy(dtype=float), g[ycol].to_numpy(dtype=float)))
                if len(verts) < 3:
                    continue    
                polygons_list.append(verts)
            
            if polygons_list:
                shapes_layer = viewer.add_shapes(
                    polygons_list,
                    shape_type='polygon',
                    edge_width=20,
                    edge_color='white',
                    face_color=[0.8, 0.8, 0.8, 0.8],
                    name=f"{region_name}"
                )
         

                print(f"  -> Added shapes for {region_name}")



In [ ]:
# Create ct_simple column: Extract simplified cell types from region-specific names
# This reduces cell types like "Astrocytes.cortex.hippocampus" to just "Astrocytes"

# Get all unique cell types
unique_cell_types = adata_full.obs['cell_type'].unique()
print(f"Found {len(unique_cell_types)} unique cell types")

# Define multi-part patterns that should be kept together
# These patterns will be preserved as-is (e.g., "Excitatory.neurons.layer.1" -> "Excitatory.neurons")
multi_part_patterns = [
    "Excitatory.neurons",
    "Inhibitory.neurons", 
    "Interneuron",
    "Interneurons",
    "Astrocytes"
]

# Define cell types that should remain unchanged (no simplification)
unchanged_types = [
    "Mature.oligodendrocytes",
    "Myelin.forming.oligodendrocytes",
    "Committed.oligodendrocytes",
    "Granule.neurons",
    "Oligodendrocyte.precursor.cells",
    "Vascular.leptomeningeal.cells",
    "Vascular.smooth.muscle.cells",
    "Astrocytes.Bergmann.glia",
    "Purkinje.cells",
    "Radial.glia",
    "Ependymal.cells",
    "Neurogliaform.cells",
    "T.cell",
    "Serotonergic.neurons",
    "Newly.formed.oligodendrocytes",
    "CCK.interneurons",
    "Dopaminergic.neurons",
    "Peptidergic.neurons",
    "Olfactory.ensheathing.cells",
    "Cajal.Retzius.cells"
]

# Define specific mappings for special cases
special_mappings = {
    # --- FIX: Ensure all Interneuron variants map to the PLURAL "Interneurons" ---
    "Interneuron": "Interneurons",             
    "Inhibitory.interneurons": "Interneurons", 
    "Interneuron.selective.interneurons": "Interneurons",
    
    # Original mappings
    "D1.medium.spiny.neurons": "Spiny.neurons",
    "D2.medium.spiny.neurons": "Spiny.neurons",
    "Cholinergic.neurons.habenula": "Cholinergic.neurons",
    "Vascular.endothelial.cells": "Endothelial.cells",
    "Telencephalon.inhibitory.neurons": "Inhibitory.neurons",
    "Choroid.plexus.epithelial.cells": "Epithelial.cells",
    "Olfactory.bulb.inhibitory.neurons": "Inhibitory.neurons",
    "Hindbrain.inhibitory.neurons": "Inhibitory.neurons",
    "Hindbrain.excitatory.neurons": "Excitatory.neurons",

}

# Simplify these by stripping suffixes (Prefix -> General)
multi_part_patterns = [
    "Excitatory.neurons",
    "Inhibitory.neurons", 
    "Interneurons",
    "Astrocytes"
    # Note: "Interneuron" (singular) removed from patterns to avoid partial matches returning singular
]


# ------------------
def extract_simple_cell_type(cell_type):
    if pd.isna(cell_type):
        return cell_type
    
    ct_str = str(cell_type)
    
    # Unchanged (Catch specific complex names first)
    if ct_str in unchanged_types:
        return ct_str
    
    # Special Mappings (Fix specific overrides BEFORE pattern matching)
    if ct_str in special_mappings:
        return special_mappings[ct_str]
    
    # Pattern Matching (Prefix)
    for pattern in multi_part_patterns:
        # Check if it IS the pattern or STARTS with pattern + dot
        if ct_str == pattern or ct_str.startswith(pattern + "."):
            return pattern

    # Catch-all for singular "Interneuron" pattern misses
    if ct_str.startswith("Interneuron."):
        return "Interneurons"
            
    # Fallback
    return ct_str

# Apply Optimization
# ---------------------
print("\nGenerating mapping dictionary...")

# Get unique types and create a lookup dictionary
unique_types = adata_full.obs['cell_type'].unique()
type_mapping_dict = {ct: extract_simple_cell_type(ct) for ct in unique_types}

# Map the dictionary to the column
adata_full.obs['ct_simple'] = adata_full.obs['cell_type'].map(type_mapping_dict)

# Validation
# -------------
print("\n✓ ct_simple column created!")
print(f"Original unique types: {len(unique_types)}")
print(f"New unique types:      {adata_full.obs['ct_simple'].nunique()}")

print("\nValue counts for ct_simple:")
print(adata_full.obs['ct_simple'].value_counts())

In [ ]:
#create bar graphs
import matplotlib.ticker as ticker
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
stroke_color = (249/255,100/255,149/255)
healthy_color=(85/255,160/255,251/255)
cntrl_color=(184/255,86/255,215/255)



strata_col='Treatment'
strata_options=['Sham','Treatment']
def plot_celltype_counts_per_FMT(adata, top_n):

    ds=adata
    counts = ds.obs.groupby([strata_col, "ct_simple"]).size()
    top_clusters = (
        counts
        .groupby(level="ct_simple")
        .sum()
        .nlargest(top_n)
        .index
    )
    df = counts.unstack(level=strata_col).loc[top_clusters].fillna(0)
    df = df[strata_options]
    print(df.sum(axis=0))
    df_percent = df.div(df.sum(axis=0), axis=1) * 100
    print(df_percent)

    ax = df_percent.plot(
        kind="bar",
        color={strata_options[0]: healthy_color, strata_options[1]: stroke_color},
        figsize=(16, 12),
        width=0.8,
    )

    ax.legend( fontsize=28, title_fontsize=30, loc="upper right")
    ax.set_ylabel("Percentage of Cell Type", fontsize=30)
    print(f"Cell Cluster Abundance: Stroke-FMT vs. Healthy-FMT")
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=20)

    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

plot_celltype_counts_per_FMT(adata_full, top_n=16)


In [ ]:
from typing import Dict, Union
import numpy as np
import pandas as pd
from anndata import AnnData
adata_full=adata_full[adata_full.obs['napari_region']!='Olfactory']
adata_full=adata_full[adata_full.obs['FMT']!='Cntrl']


random_state=0
def alloc_by_baseline_with_caps(baseline_counts: pd.Series, caps: pd.Series, total: int) -> pd.Series:
    """
    Allocate 'total' items across indices in proportion to 'baseline_counts',
    without exceeding 'caps' (remaining available per region).
    Simple proportional allocation + small fix if caps bind.
    """
    out = pd.Series(0, index=baseline_counts.index, dtype=int)
    remain = int(total)
    # weights from baseline (relative proportions)
    w = baseline_counts.clip(lower=0).astype(float)
    if w.sum() == 0 or remain <= 0:
        return out

    # 1st pass: proportional
    raw = w / w.sum() * remain
    base = np.floor(raw).astype(int)

    # respect caps
    take = base.clip(upper=caps.astype(int))
    out += take
    remain -= int(take.sum())

    # 2nd pass: give leftover to the largest fractional remainders, respecting caps
    if remain > 0:
        frac = (raw - base).sort_values(ascending=False)
        for idx in frac.index:
            if remain == 0:
                break
            if out[idx] < int(caps.get(idx, 0)):
                out[idx] += 1
                remain -= 1
    return out

def set_region_abundance_by_FMT(
    adata: AnnData,
    target_by_fmt: Dict[str, Dict[str, Union[int, float]]],
    total_per_fmt: Union[int, Dict[str, int], None] = None,
    region_col: str = "napari_region",
    fmt_col: str = "FMT",
    random_state: int | None =0,
) -> AnnData:
    """
    Select cells per FMT to match targets for specified regions (counts or fractions).
    All unspecified regions keep their *relative proportions* (baseline mix within that FMT),
    subject to availability caps. Ensures per-FMT totals are <= t and never exceed availability.
    """
    rng = np.random.default_rng(random_state)

    # Use existing labels only; avoid turning NaNs into "nan" strings
    obs_rf = adata.obs[[region_col, fmt_col]].dropna().copy()

    # availability table: counts per FMT x region
    avail = obs_rf.groupby([fmt_col, region_col]).size().unstack(fill_value=0)

    def _alloc_by_baseline_with_caps(baseline_counts: pd.Series,
                                     caps: pd.Series,
                                     total: int) -> pd.Series:
        """
        Allocate 'total' across indices proportionally to baseline_counts, without exceeding caps.
        Two-pass: floor proportional then distribute leftover by largest remainders.
        """
        out = pd.Series(0, index=baseline_counts.index, dtype=int)
        total = int(total)
        if total <= 0 or baseline_counts.sum() <= 0 or caps.sum() <= 0:
            return out

        weights = baseline_counts.clip(lower=0).astype(float)
        weights = weights / (weights.sum() if weights.sum() > 0 else 1.0)

        raw = weights * total
        base = np.floor(raw).astype(int)
        take = base.clip(upper=caps.astype(int))
        out += take
        rem = total - int(take.sum())
        if rem <= 0:
            return out

        frac = (raw - base).sort_values(ascending=False)
        for idx in frac.index:
            if rem == 0:
                break
            if out[idx] < int(caps.get(idx, 0)):
                out[idx] += 1
                rem -= 1
        return out

    selected = []

    for fmt in avail.index:
        row = avail.loc[fmt]                     # available counts per region for this FMT
        total_avail = int(row.sum())

        # choose per-FMT target total t
        if isinstance(total_per_fmt, dict):
            t = int(min(total_per_fmt.get(fmt, total_avail), total_avail))
        elif isinstance(total_per_fmt, int):
            t = int(min(total_per_fmt, total_avail))
        else:
            t = total_avail

        # baseline mix for unspecified regions (keep-as-is proportions)
        baseline = (row / row.sum()).fillna(0)

        desired = pd.Series(0, index=row.index, dtype=int)

        if fmt in target_by_fmt:
            tgt = target_by_fmt[fmt]

            # Determine if targets are explicit counts or fractions
            is_count = all(isinstance(v, (int, np.integer)) for v in tgt.values())

            if is_count:
                # 1) lock specified counts
                for r, v in tgt.items():
                    if r in desired.index:
                        desired[r] = int(v)
            else:
                # 1) lock specified fractions
                frac_map = {r: float(v) for r, v in tgt.items() if r in desired.index}
                sfrac = sum(frac_map.values())
                if sfrac > 1.0 and sfrac > 0:
                    frac_map = {r: v / sfrac for r, v in frac_map.items()}
                for r, fr in frac_map.items():
                    desired[r] = int(round(fr * t))

            # Cap specified to availability
            desired = desired.clip(upper=row)

            # 2) allocate leftover to unspecified regions by baseline ratios (respect caps)
            unspecified = [r for r in desired.index if r not in tgt]
            remain = max(0, t - int(desired.sum()))
            if unspecified and remain > 0:
                caps_unspec = (row - desired).loc[unspecified].clip(lower=0)
                alloc = _alloc_by_baseline_with_caps(baseline.loc[unspecified], caps_unspec, remain)
                desired.loc[unspecified] += alloc

        else:
            # No targets: keep baseline mix up to t (respect caps)
            caps_all = row.clip(lower=0)
            desired = _alloc_by_baseline_with_caps(row, caps_all, t)

        # Safety: never over t (small rounding trims)
        while int(desired.sum()) > t:
            biggest = desired.idxmax()
            if desired[biggest] == 0:
                break
            desired[biggest] -= 1

        # Final cap to availability
        desired = desired.clip(upper=row)

        # Sample the requested cells for this FMT
        sub = obs_rf[obs_rf[fmt_col] == fmt]
        for region, need in desired.items():
            n = int(need)
            if n <= 0:
                continue
            pool = sub.index[sub[region_col] == region].to_numpy()
            if n >= len(pool):
                selected.extend(pool.tolist())
            else:
                selected.extend(rng.choice(pool, size=n, replace=False).tolist())

    sel_set = set(selected)
    sel_index = [idx for idx in obs_rf.index if idx in sel_set]
    if not sel_index:
        raise ValueError("No cells selected.")

    return adata[sel_index].copy()

#()()()()()()()()()()()()()(()()()()())()()()()()()()()()()()()()()()()()()()()()()()()()()()()
#()()()()()()()()()()()()()(()()()()())()()()()()()()()()()()()()()()()()()()()()()()()()()()()


sample_col = "sample_ID"  
region_col = "napari_region"
# 1. Calculate the fraction of cells per region for each sample
# Counts per sample per region
counts_per_sample = adata_full.obs.groupby([sample_col, region_col]).size().unstack(fill_value=0)
# Convert to fractions (row-wise normalization)
fractions_per_sample = counts_per_sample.div(counts_per_sample.sum(axis=1), axis=0)

# 2. Calculate Mean and STD across samples
region_means = fractions_per_sample.mean()
region_stds = fractions_per_sample.std()

# 3. Define helper variables dynamically
# Dictionary to store stats for easy access if needed
stats = {}
regions_of_interest = ["Cortex", "Cerebellum", "Hippocampus"]

# Initialize variables with calculated values
cortex_n = region_means.get("Cortex", 0.0)
cortex_h = cortex_n + region_stds.get("Cortex", 0.0)
cortex_l = max(0.0, cortex_n - region_stds.get("Cortex", 0.0))

cerebellum_n = region_means.get("Cerebellum", 0.0)
cerebellum_h = cerebellum_n + region_stds.get("Cerebellum", 0.0)
cerebellum_l = max(0.0, cerebellum_n - region_stds.get("Cerebellum", 0.0))

hippocampus_n = region_means.get("Hippocampus", 0.0)
hippocampus_h = hippocampus_n + region_stds.get("Hippocampus", 0.0)
hippocampus_l = max(0.0, hippocampus_n - region_stds.get("Hippocampus", 0.0))



mean_cort=region_stds.get( "Cortex", 0.0)
print("--- Calculated Stats (Fractions) ---")
print(f"Cortex:      Mean={cortex_n:.4f}, STD={mean_cort:.4f}, High={cortex_h:.4f}, Low={cortex_l:.4f}")
print(f"Cerebellum:  Mean={cerebellum_n:.4f}, High={cerebellum_h:.4f}, Low={cerebellum_l:.4f}")
print(f"Hippocampus: Mean={hippocampus_n:.4f}, High={hippocampus_h:.4f}, Low={hippocampus_l:.4f}")

#()()()()()()()()()()()()()(()()()()())()()()()()()()()()()()()()()()()()()()()()()()()()()()()
# TARGET DEFINITIONS
#()()()()()()()()()()()()()(()()()()())()()()()()()()()()()()()()()()()()()()()()()()()()()()()


target_by_Cortex_up = {
    "Stroke_FMT": {"Cortex": cortex_h},  
    "Healthy_FMT": {"Cortex": cortex_l},
    "Cntrl": {"Cortex": cortex_n},   
}
target_by_Cortex_down = {
    "Stroke_FMT": {"Cortex": cortex_l},   
    "Healthy_FMT": {"Cortex": cortex_h},
    "Cntrl": {"Cortex": cortex_n}, 
}

#EXCLUDING BRAINSTEM
adata_full=adata_full[adata_full.obs['napari_region']!='Olfactory']

print(cortex_h)
print(cortex_n)
print(cortex_l)

cortex_up = set_region_abundance_by_FMT(adata_full, target_by_Cortex_up, total_per_fmt=125000, random_state=0)
cortex_down = set_region_abundance_by_FMT(adata_full, target_by_Cortex_down, total_per_fmt=125000, random_state=0)



In [ ]:
written_files = export_anndata_for_seurat(
     adata_full,#this Adata_full has had the Olfactory bulb and Cntrl group removed.
     r"DE_csv_folder\base"
)
written_files = export_anndata_for_seurat(
     cortex_up,
     r"DE_csv_folder\up"
 )
written_files = export_anndata_for_seurat(
     cortex_down,
     r"DE_csv_folder\down"
 )

